In [ ]:
import os
import requests
import json
import pandas as pd
import mysql.connector as connector
from dotenv import load_dotenv


In [ ]:
load_dotenv(".env")

In [ ]:
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")



In [ ]:
SEASON = 2024
LEAGUE_ID = 39

# ------------------------EXTRACT------------------------

In [ ]:
# Accessing database and collecting data
URL = "https://v3.football.api-sports.io/standings"

headers = {
    'x-apisports-key': API_KEY
    }
querystring ={
    "league" : LEAGUE_ID,
    "season" : SEASON
}

response = requests.get(
    url=URL,
    headers = headers,
    params = querystring
)
payload = response.json()



In [ ]:
formatted_data = json.dumps(payload,indent=2) # enhances readability. Converts json to string - dump string
print(formatted_data)

In [ ]:
payload["response"]

In [ ]:
standings_list = payload["response"][0]["league"]["standings"][0]
formatted_standings = json.dumps(standings_list, indent = 2)
print(formatted_standings)
print(type(standings_list))
print(type(formatted_standings))

In [ ]:
standings_list


# --------------TRANSFORM----------------------

In [ ]:
count = 0
for item in standings_list:
    count+=1
print(count)

In [ ]:
standings_list[0]

In [ ]:
print(standings_list[0].keys())

In [ ]:
standings_rows = []
for idx,club in enumerate(standings_list):
    season          = 2024
    position        = club['rank']
    team_id         = club['team']['id']
    team            = club['team']['name']
    points          = club['points']
    played          = club['all']['played']
    won             = club['all']['win']
    draw            = club['all']['draw']
    lost            = club['all']['lose']
    goals_for       = club['all']['goals']['for']
    goals_against   = club['all']['goals']['against']
    goals_diff      = club['goalsDiff']
    form            = club['form']

     ##active_row = (season, position, team_id, team, points, 
     #              played, won, draw, lost, goals_for,
      #              goals_against, goals_diff, form
       #             )
    
    active_row = {
        'season':season, 
        'position':position,
        'team_id':team_id, 
        'team':team,
        'points':points, 
        'played':played, 
        'won':won,
        'draw': draw,
        'lost':lost, 
        'goals_for':goals_for,
        'goals_against':goals_against,
        'goals_diff':goals_diff, 
        'form':form
    }
    standings_rows.append(active_row)
print(standings_rows)


In [ ]:
df = pd.DataFrame(standings_rows)
df.keys()

In [ ]:
df.info()

# ----------------- LOAD---------------

In [ ]:
# MYSQL CREDENTIALS
MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_DATABASE_NAME = os.getenv("MYSQL_DATABASE_NAME")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_USER =os.getenv("MYSQL_USER")

In [ ]:
server_conn = connector.connect(
    host = MYSQL_HOST,
    port = MYSQL_PORT,
    user = MYSQL_USER,
    password = MYSQL_PASSWORD,
    connection_timeout = 10,
    autocommit = False,
    raise_on_warnings = True # pythonic error handling
    
)
server_conn_cur = server_conn.cursor()
print(f'Success - Connected to mySQL server!')

In [ ]:
server_conn_cur.close()
server_conn.close()

In [ ]:
db_conn = connector.connect(
    host = MYSQL_HOST,
    port = MYSQL_PORT,
    user = MYSQL_USER,
    password = MYSQL_PASSWORD,
    database = MYSQL_DATABASE_NAME
)
db_conn_cursor = db_conn.cursor()
print(f'Success - Connected to mySQL database!')

In [ ]:
sql_table = "standings"
db_conn_cursor.execute("SHOW TABLES LIKE %s", (f'{sql_table}',))

if db_conn_cursor.fetchone() is None:
    raise SystemExit(f"Table {sql_table} could Not be found. You may need to create it")
else:
    print(f'Success - {sql_table} exists!')

In [ ]:
column_names = ['season', 'position', 'team_id', 'team', 'points', 'played', 'won',
       'draw', 'lost', 'goals_for', 'goals_against', 'goals_diff', 'form']
data = list(
    df[column_names].itertuples(index=False, name=None)
)

In [ ]:
sql=f"""
INSERT INTO {sql_table}
(season, position, team_id, team, points, played, won,
       draw, lost, goals_for, goals_against, goals_diff, form)
VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) AS src
ON DUPLICATE KEY UPDATE
position      = src.position,
team          = src.team,
played        = src.played,
won           = src.won,
draw          = src.draw,
lost          = src.lost,
goals_for     = src.goals_for,
goals_against = src.goals_against,
goals_diff    = src.goals_diff,
points        = src.points,
form          = src.form

"""

In [ ]:
try:
    db_conn_cursor.executemany(sql, data)
    db_conn.commit()
    print(f'Success - {db_conn_cursor.rowcount} rows affected')
except connector.Error as err:
    db_conn.rollback()
    print(f'Database Error: {err}')
finally:
    db_conn_cursor.close()
    db_conn.close()
    print("All database connections are now closed!")